# PPO Overtake Lab

This notebook is the interactive version of the PPO highway overtaking setup.

Use it to:
- tweak reward shaping for overtaking
- change traffic difficulty and steering limits
- train PPO runs
- inspect evaluation metrics
- render human-view rollouts

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

from stable_baselines3 import PPO


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "ppo"
SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "Elurant_PPO"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from ppo_overtake_lab import (
    LAB_OUTPUT_DIR,
    ExperimentConfig,
    PPOConfig,
    RewardConfig,
    ScenarioConfig,
    evaluate_model,
    lane_density_override,
    render_human_episodes,
    train_overtake_agent,
)

print("Notebook dir:", NOTEBOOK_DIR)
print("Script dir:", SCRIPT_DIR)
print("Output dir:", LAB_OUTPUT_DIR)


## 1. Experiment Knobs

This is the main cell you should edit.

The defaults below are intentionally overtaking-focused rather than paper-faithful:
- slower traffic ahead so overtaking is feasible
- stronger reward on overtaking progress and final success
- smaller steering range and higher control frequency for smoother lane changes

In [ ]:
experiment = ExperimentConfig(
    name='overtake_push_v1',
    reward=RewardConfig(
        collision_penalty=-120.0,
        offroad_penalty=-120.0,
        speed_weight=28.0,
        progress_bonus=24.0,
        success_bonus=100.0,
        steering_penalty=0.35,
        blocked_in_right_penalty=-4.0,
        blocked_overtake_bonus=4.5,
        keep_right_bonus=1.5,
        unsafe_headway_penalty=-8.0,
        normalize_reward=False,
    ),
    scenario=ScenarioConfig(
        lanes_count=3,
        vehicles_per_lane=5,
        observation_vehicles=16,
        duration=50,
        simulation_frequency=20,
        policy_frequency=2,
        steering_range=0.01,
        ego_speed_range=(24.0, 26.0),
        other_speed_range=(18.0, 22.0),
        spawn_lead_range=(60.0, 110.0),
        spawn_gap_range=(25.0, 45.0),
        initial_lane_id=2,
    ),
    ppo=PPOConfig(
        timesteps=30000,
        learning_rate=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        batch_size=256,
        n_steps=512,
        n_epochs=10,
        ent_coef=0.01,
        vf_coef=0.5,
        n_envs=24,
        eval_freq=5000,
        eval_episodes=10,
        seed=42,
        policy_pi=(256, 256),
        policy_vf=(256, 256),
    ),
)

print(json.dumps(asdict(experiment), indent=2))

## 2. Train PPO

Run this cell after editing the config above. It saves the run under `artifacts/ppo/ppo_overtake_lab/<experiment.name>/`.

In [ ]:
model, evaluation = train_overtake_agent(experiment)
print(json.dumps(evaluation, indent=2))

## 3. Reload Best Model Later

Use this cell if you restart the notebook and want to reuse the saved checkpoint.

In [ ]:
best_model_path = LAB_OUTPUT_DIR / experiment.name / 'best_model' / 'best_model.zip'
final_model_path = LAB_OUTPUT_DIR / experiment.name / 'final_model.zip'
load_path = best_model_path if best_model_path.exists() else final_model_path
model = PPO.load(str(load_path))
print('Loaded:', load_path)

## 4. Evaluate Base + Harder Traffic Scenarios

In [ ]:
base_eval = evaluate_model(model, experiment, episodes=10, base_seed=10000)
adapt_eval_dense = evaluate_model(
    model,
    experiment,
    episodes=10,
    base_seed=20000,
    config_overrides=lane_density_override(lanes_count=3, vehicles_per_lane=7),
)
adapt_eval_narrow = evaluate_model(
    model,
    experiment,
    episodes=10,
    base_seed=30000,
    config_overrides=lane_density_override(lanes_count=2, vehicles_per_lane=8),
)

print('Base:')
print(json.dumps(base_eval, indent=2))
print('\nDense traffic:')
print(json.dumps(adapt_eval_dense, indent=2))
print('\nNarrow highway:')
print(json.dumps(adapt_eval_narrow, indent=2))

## 5. Human Rendering

This opens a live window so you can actually watch the policy overtake.

In [ ]:

render_human_episodes(model, experiment, episodes=3, sleep=0.05)

## 6. Suggested Tuning Directions

If the agent is too passive:
- increase `progress_bonus`
- increase `blocked_overtake_bonus`
- lower `keep_right_bonus`
- make `other_speed_range` slower

If the agent is too reckless:
- increase `unsafe_headway_penalty`
- increase `steering_penalty`
- reduce `steering_range`
- increase `policy_frequency`

If the agent drives safely but never finishes overtakes:
- increase `success_bonus`
- increase `timesteps`
- increase `n_envs`
- widen the speed gap between ego and traffic